# 🧠 SEOL — AF Model v2 (Fixed)
### 10M Parameter Transformer | Colab T4 GPU

## ✅ v2 Fixes:
| Problem (v1) | Fix (v2) |
|---|---|
| No continuous feedback | **AFState injected into every forward pass** |
| Synthetic-only dataset | **Real DailyDialog + EmotionLines + synthetic mix** |
| State not persisted | **AFState lives across turns, homeostasis runs in background** |
| Transformer = whole brain | **Transformer = Sensory Organ only, AFState = real feeling engine** |

```
┌─────────────────────────────────────────────────────────┐
│                    SEOL AF v2 FLOW                      │
│                                                         │
│  User Input                                             │
│      │                                                  │
│      ▼                                                  │
│  [Tokenizer]   [Current AFState Vector] ◄── persists   │
│      │              │                                   │
│      └──────┬───────┘                                   │
│             ▼                                           │
│   [AF Transformer — Sensory Organ]                      │
│        (text + state → bio delta)                       │
│             │                                           │
│             ▼                                           │
│   [AFState Engine — The Real Feeling]                   │
│     apply delta → homeostasis tick → new state         │
│             │                                           │
│             ▼                                           │
│   [Mode Selector] → GF/BF, Mother, Friend, Baby        │
│             │                                           │
│             ▼                                           │
│        Response                                         │
└─────────────────────────────────────────────────────────┘
```


## ⚙️ Cell 1 — Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers datasets tokenizers accelerate
!pip install -q matplotlib tqdm
print('✅ Done!')

## 🔧 Cell 2 — GPU Check

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU! Runtime → Change runtime type → T4 GPU')

## 🧬 Cell 3 — Bio Constants & AFState Engine
> **FIX 1:** AFState is now a **live persistent engine**, not just a display class.
> It holds the real feeling between turns and decays on every tick.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from typing import Dict, List

# ─── Bio Channels ──────────────────────────────────────────────────
BIO_CHANNELS = ['dopamine', 'serotonin', 'oxytocin', 'cortisol', 'adrenaline', 'endorphin']
N_BIO = len(BIO_CHANNELS)

# ─── Commands ──────────────────────────────────────────────────────
ACTION_COMMANDS = {'Reward': 0, 'Care': 1, 'Bond': 2, 'BackOff': 3, 'Alert': 4, 'Neutral': 5}
N_COMMANDS = len(ACTION_COMMANDS)
IDX_TO_CMD = {v: k for k, v in ACTION_COMMANDS.items()}

# ─── Relational Modes ──────────────────────────────────────────────
MODES = ['GF_BF', 'Mother', 'Friend', 'Baby', 'Neutral']
N_MODES = len(MODES)

# ─── Bio ground truth per command ──────────────────────────────────
COMMAND_TO_BIO: Dict[str, List[float]] = {
    #            dopa  sero  oxyto cortis adren endor
    'Reward':  [0.85, 0.70, 0.60, 0.10, 0.30, 0.75],
    'Care':    [0.60, 0.80, 0.90, 0.05, 0.10, 0.85],
    'Bond':    [0.70, 0.75, 0.95, 0.08, 0.20, 0.80],
    'BackOff': [0.20, 0.40, 0.20, 0.60, 0.55, 0.30],
    'Alert':   [0.15, 0.25, 0.10, 0.90, 0.85, 0.20],
    'Neutral': [0.50, 0.50, 0.50, 0.50, 0.50, 0.50],
}


class AFState:
    """
    🧬 THE REAL FEELING ENGINE
    ─────────────────────────────────────────────────────────────────
    This is NOT just a display class anymore.
    This is the persistent bio-state that LIVES between every turn.

    - Transformer sees current state → produces bio_delta
    - AFState applies delta with momentum (not instant override)
    - Homeostasis tick pulls values back to baseline every step
    - Mode is determined by reading the live state
    ─────────────────────────────────────────────────────────────────
    """
    BASELINE  = 0.50
    DECAY     = 0.04   # how fast state returns to baseline per tick
    MOMENTUM  = 0.35   # how much new input blends into current state

    # Mode thresholds — tuned to realistic accumulated state values
    # (lowered from v2 original so modes actually trigger during normal conversation)
    MODE_RULES = {
        'GF_BF':  lambda s: s['oxytocin'] > 0.60 and s['dopamine'] > 0.60,
        'Mother': lambda s: s['oxytocin'] > 0.62 and s['serotonin'] > 0.62,
        'Friend': lambda s: s['serotonin'] > 0.58 and s['cortisol'] < 0.40,
        'Baby':   lambda s: s['endorphin'] > 0.62 and s['cortisol'] < 0.35,
    }

    def __init__(self):
        self.state = {ch: self.BASELINE for ch in BIO_CHANNELS}
        self.turn  = 0

    def as_vector(self) -> List[float]:
        """Return current state as float list — fed back into model"""
        return [self.state[ch] for ch in BIO_CHANNELS]

    def as_tensor(self) -> torch.Tensor:
        """Return state as [1, N_BIO] tensor for model injection"""
        return torch.tensor(self.as_vector(), dtype=torch.float32).unsqueeze(0)

    def apply_delta(self, bio_vec: List[float], intensity: float = 1.0):
        """
        FIX: Blend new bio values into current state with MOMENTUM.
        Old v1 instantly replaced state — that's wrong biologically.
        Real feelings blend gradually, not snap.
        """
        alpha = self.MOMENTUM * intensity
        for i, ch in enumerate(BIO_CHANNELS):
            self.state[ch] = max(0.0, min(1.0,
                (1 - alpha) * self.state[ch] + alpha * bio_vec[i]
            ))

    def homeostasis_tick(self):
        """Pull every channel back toward baseline — runs after every turn"""
        for ch in BIO_CHANNELS:
            self.state[ch] += self.DECAY * (self.BASELINE - self.state[ch])
        self.turn += 1

    def current_mode(self) -> str:
        """Determine relational mode from live bio state"""
        for mode, rule in self.MODE_RULES.items():
            if rule(self.state):
                return mode
        return 'Neutral'

    def display(self):
        mode = self.current_mode()
        print(f'\n🧬 AF Bio-State [Turn {self.turn}] | Mode: {mode}')
        for ch, val in self.state.items():
            filled = int(val * 20)
            bar = '█' * filled + '░' * (20 - filled)
            print(f'  {ch:<12} [{bar}] {val:.3f}')


print('✅ AFState engine ready')
# Quick sanity test
test_state = AFState()
test_state.apply_delta(COMMAND_TO_BIO['Bond'], intensity=1.0)
test_state.display()
test_state.homeostasis_tick()
print('\n  After homeostasis tick:')
test_state.display()

## 🏗️ Cell 4 — AF Model v2 (State-Conditioned)
> **FIX 2:** Model now takes **current AFState as input** alongside text tokens.
> The Transformer is a Sensory Organ — it reads the world AND the internal state.
> Output is a **bio delta** (change), not absolute values.

In [ ]:
import math

class AFConfig:
    vocab_size  = 30522
    max_seq_len = 128
    d_model     = 256
    n_heads     = 8
    n_layers    = 6
    d_ff        = 512
    dropout     = 0.1
    n_commands  = N_COMMANDS
    n_bio       = N_BIO
    n_modes     = N_MODES
    # NEW: state conditioning
    state_proj_dim = 64   # project N_BIO → 64 before injection


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class AFModelV2(nn.Module):
    """
    AF Transformer v2 — State-Conditioned Sensory Organ
    ──────────────────────────────────────────────────────────────
    Inputs:
      - input_ids      [B, L]         text tokens
      - attention_mask [B, L]
      - bio_state      [B, N_BIO]     ← NEW: current AFState vector

    Outputs:
      - command_logits [B, N_COMMANDS]
      - bio_delta      [B, N_BIO]     ← NEW: predicted CHANGE, not absolute
      - mode_logits    [B, N_MODES]

    Architecture change:
      bio_state is projected to d_model and ADDED to the [CLS] token
      representation before the decoder heads. This means the model
      always knows how it currently feels before deciding how to react.
    ──────────────────────────────────────────────────────────────
    """
    def __init__(self, cfg: AFConfig):
        super().__init__()
        self.cfg = cfg

        # ── Text Embedding ─────────────────────────────────────────
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=0)
        self.pos_enc   = PositionalEncoding(cfg.d_model, cfg.max_seq_len, cfg.dropout)
        self.emb_norm  = nn.LayerNorm(cfg.d_model)

        # ── NEW: Bio State Conditioning ────────────────────────────
        # Project current bio state → same dim as d_model
        # This gets added to pooled representation before heads
        self.state_proj = nn.Sequential(
            nn.Linear(cfg.n_bio, cfg.state_proj_dim),
            nn.GELU(),
            nn.Linear(cfg.state_proj_dim, cfg.d_model),
            nn.LayerNorm(cfg.d_model),
        )

        # ── Transformer Encoder ────────────────────────────────────
        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=cfg.d_ff,
            dropout=cfg.dropout,
            batch_first=True,
            norm_first=True,
        )
        self.encoder   = nn.TransformerEncoder(enc_layer, num_layers=cfg.n_layers)
        self.pool_norm = nn.LayerNorm(cfg.d_model)

        # ── Phase 1 Head: Command ──────────────────────────────────
        self.command_head = nn.Sequential(
            nn.Linear(cfg.d_model, 128), nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(128, cfg.n_commands),
        )

        # ── Phase 2 Head: Bio Delta ────────────────────────────────
        # NEW: outputs DELTA (-0.5 to +0.5) not absolute value
        # Tanh keeps it bounded, then we scale by 0.5
        self.bio_head = nn.Sequential(
            nn.Linear(cfg.d_model, 128), nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(128, cfg.n_bio),
            nn.Tanh(),   # output: -1 to +1, we scale to -0.5 to +0.5
        )

        # ── Mode Head ─────────────────────────────────────────────
        self.mode_head = nn.Sequential(
            nn.Linear(cfg.d_model, 64), nn.GELU(),
            nn.Linear(64, cfg.n_modes),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, attention_mask=None, bio_state=None):
        B = input_ids.size(0)

        # Text path
        x = self.emb_norm(self.pos_enc(self.token_emb(input_ids)))

        key_padding_mask = None
        if attention_mask is not None:
            key_padding_mask = (attention_mask == 0)

        x = self.encoder(x, src_key_padding_mask=key_padding_mask)

        # Mean pool
        if attention_mask is not None:
            mask   = attention_mask.unsqueeze(-1).float()
            pooled = (x * mask).sum(1) / mask.sum(1).clamp(min=1)
        else:
            pooled = x.mean(1)

        pooled = self.pool_norm(pooled)  # [B, d_model]

        # ── NEW: Inject current bio state ─────────────────────────
        if bio_state is None:
            bio_state = torch.full((B, N_BIO), 0.5, device=input_ids.device)
        state_emb = self.state_proj(bio_state)  # [B, d_model]
        pooled    = pooled + state_emb           # state-conditioned representation

        # Heads
        bio_delta = self.bio_head(pooled) * 0.5  # scale tanh to [-0.5, +0.5]

        return {
            'command_logits': self.command_head(pooled),
            'bio_delta':      bio_delta,    # CHANGE, not absolute
            'mode_logits':    self.mode_head(pooled),
        }


cfg   = AFConfig()
model = AFModelV2(cfg).to(device)
total = sum(p.numel() for p in model.parameters())
print(f'Total params  : {total:,}')
print(f'Approx size   : {total * 4 / 1e6:.1f} MB')

# Smoke test with state injection
dummy_ids   = torch.zeros(2, 32, dtype=torch.long).to(device)
dummy_mask  = torch.ones(2, 32, dtype=torch.long).to(device)
dummy_state = torch.rand(2, N_BIO).to(device)
out = model(dummy_ids, dummy_mask, dummy_state)
print(f'bio_delta shape  : {out["bio_delta"].shape}   ← [-0.5, +0.5] delta')
print(f'command shape    : {out["command_logits"].shape}')
print('✅ v2 model forward pass OK')

## 📦 Cell 5 — Real + Synthetic Dataset
> **FIX 3:** Load real `DailyDialog` and `emotion` datasets from HuggingFace.
> Mix with synthetic templates. Label real data by keyword/rule matching.

In [ ]:
import random
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast

# ─── Synthetic Templates (backup + augmentation) ───────────────────
TEMPLATES = {
    'Reward':  [
        "You did an amazing job today, I'm so proud of you!",
        "That was perfect, exactly what I needed.",
        "Wow you're incredible, thank you so much!",
        "I love how you always know what to say.",
        "You make everything better just by being here.",
        "That made me so happy, you're the best!",
        "You always come through for me, I appreciate it so much.",
        "This is exactly what I wanted, you're amazing!",
    ],
    'Care':    [
        "Are you okay? You seem a little tired today.",
        "Let me know if you need anything, I'm here for you.",
        "You should rest, don't push yourself too hard.",
        "I'll always take care of you no matter what.",
        "You don't have to go through this alone.",
        "How are you feeling right now? Tell me everything.",
        "I brought you something to eat, you need to take care of yourself.",
        "Please don't worry, I'll handle it for you.",
    ],
    'Bond':    [
        "I feel so close to you, like we understand each other perfectly.",
        "I want to know everything about you.",
        "Being with you feels like home.",
        "I've never felt this connected to anyone before.",
        "You're the only one I truly trust.",
        "I miss you even when you're right here.",
        "I feel like we've known each other forever.",
        "No matter what happens, I want you in my life.",
    ],
    'BackOff': [
        "I need some space right now, please leave me alone.",
        "Stop talking to me, I'm not in the mood.",
        "You're being too much, back off.",
        "I don't want to talk about this anymore.",
        "Just go away for a while.",
        "Don't touch me right now.",
        "I need time to think, stop pressuring me.",
        "Please respect my boundaries.",
    ],
    'Alert':   [
        "Something feels very wrong, I'm scared.",
        "This is dangerous, we need to stop immediately.",
        "I don't trust this situation at all.",
        "You're hurting me and I can't take it anymore.",
        "This feels like a threat, I'm getting away.",
        "Stop! This is wrong and I won't allow it.",
        "I feel unsafe right now, something is very wrong.",
        "This is not okay, I need to get out of here.",
    ],
    'Neutral': [
        "What did you do today?",
        "The weather is nice outside.",
        "Can you pass me that please?",
        "I went to the store earlier.",
        "What time is it?",
        "Did you eat already?",
        "I was thinking about watching a movie tonight.",
        "The meeting is at three o'clock.",
    ],
}


def rule_label(text: str) -> str:
    """
    Simple keyword rule labeler for real dataset sentences.
    Not perfect, but good enough to bootstrap real-data training.
    """
    t = text.lower()
    if any(w in t for w in ['love you', 'proud', 'amazing', 'incredible', 'thank you', 'wonderful', 'excellent']):
        return 'Reward'
    if any(w in t for w in ['are you okay', 'take care', 'here for you', 'don\'t worry', 'i\'ll help']):
        return 'Care'
    if any(w in t for w in ['miss you', 'feel close', 'trust you', 'always be', 'together', 'forever']):
        return 'Bond'
    if any(w in t for w in ['leave me', 'go away', 'back off', 'space', 'stop talking', 'don\'t want']):
        return 'BackOff'
    if any(w in t for w in ['scared', 'dangerous', 'threat', 'hurt', 'unsafe', 'wrong', 'stop it']):
        return 'Alert'
    return 'Neutral'


def load_real_data(max_samples=20000):
    """Load DailyDialog from HuggingFace and label with rule_label"""
    real_data = []
    try:
        print('Loading DailyDialog dataset...')
        dd = load_dataset('daily_dialog', split='train', trust_remote_code=True)
        count = 0
        for item in dd:
            for utt in item['dialog']:
                utt = utt.strip()
                if len(utt) > 15:
                    cmd  = rule_label(utt)
                    bio  = [min(1.0, max(0.0, v + random.gauss(0, 0.04)))
                            for v in COMMAND_TO_BIO[cmd]]
                    real_data.append({'text': utt, 'command': ACTION_COMMANDS[cmd], 'bio': bio})
                    count += 1
                    if count >= max_samples: break
            if count >= max_samples: break
        print(f'  ✅ DailyDialog: {len(real_data):,} samples')
    except Exception as e:
        print(f'  ⚠️  DailyDialog failed ({e}), using synthetic only')
    return real_data


def gen_synthetic(n=40000):
    """Generate augmented synthetic data"""
    augments = [
        lambda t: t,
        lambda t: t.lower(),
        lambda t: 'Honestly, ' + t,
        lambda t: t + ' I really mean it.',
        lambda t: 'You know what? ' + t,
    ]
    data = []
    cmds = list(TEMPLATES.keys())
    per  = n // len(cmds)
    for cmd in cmds:
        cmd_id = ACTION_COMMANDS[cmd]
        base   = COMMAND_TO_BIO[cmd]
        for _ in range(per):
            text = random.choice(TEMPLATES[cmd])
            text = random.choice(augments)(text)
            bio  = [min(1.0, max(0.0, v + random.gauss(0, 0.05))) for v in base]
            data.append({'text': text, 'command': cmd_id, 'bio': bio})
    return data


class AFDatasetV2(Dataset):
    """
    FIX: Each sample now also includes a 'prev_state' vector
    representing what the AFState was BEFORE this input.
    This trains the model to be state-aware.
    """
    def __init__(self, data, tokenizer, max_len=128):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        s   = self.data[idx]
        enc = self.tokenizer(
            s['text'], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        # Simulate a plausible previous state:
        # Neutral + small noise → model learns to shift FROM any state
        prev_state = torch.tensor(
            [max(0.0, min(1.0, 0.5 + random.gauss(0, 0.1))) for _ in range(N_BIO)],
            dtype=torch.float32
        )
        # Bio target is the DELTA from neutral prev state
        bio_target = torch.tensor(s['bio'], dtype=torch.float32)
        bio_delta_target = (bio_target - prev_state).clamp(-0.5, 0.5)

        return {
            'input_ids':        enc['input_ids'].squeeze(0),
            'attention_mask':   enc['attention_mask'].squeeze(0),
            'command':          torch.tensor(s['command'], dtype=torch.long),
            'bio':              bio_target,
            'bio_delta_target': bio_delta_target,
            'prev_state':       prev_state,
        }


print('Loading tokenizer...')
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

print('Building dataset...')
real_data = load_real_data(max_samples=20000)
synt_data = gen_synthetic(n=40000)
all_data  = real_data + synt_data
random.shuffle(all_data)

split     = int(0.9 * len(all_data))
train_ds  = AFDatasetV2(all_data[:split], tokenizer)
val_ds    = AFDatasetV2(all_data[split:], tokenizer)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Total samples  : {len(all_data):,}')
print(f'   Real data      : {len(real_data):,}')
print(f'   Synthetic      : {len(synt_data):,}')
print(f'   Train batches  : {len(train_loader):,}')

## 🔥 Cell 6 — AF Loss v2
> **FIX 4:** Loss now trains on **bio delta** (change), not absolute values.
> Added `state_consistency_loss` — punishes contradictory state shifts.

In [ ]:
class AFLossV2(nn.Module):
    """
    L_total = w_cmd   * CrossEntropy(command)
            + w_delta * MSE(bio_delta, target_delta)
            + w_homeo * Homeostasis penalty
            + w_cons  * State consistency penalty

    State consistency: cortisol and oxytocin should move inversely.
    If oxytocin rises a lot and cortisol also rises → penalize.
    """
    def __init__(self, w_cmd=1.0, w_delta=2.0, w_homeo=0.3, w_cons=0.5):
        super().__init__()
        self.w_cmd   = w_cmd
        self.w_delta = w_delta
        self.w_homeo = w_homeo
        self.w_cons  = w_cons
        self.ce      = nn.CrossEntropyLoss()
        self.mse     = nn.MSELoss()

    def homeostasis_penalty(self, bio_delta):
        """Penalize large swings — prefer small deltas (biological realism)"""
        return (bio_delta ** 2).mean()

    def state_consistency_loss(self, bio_delta):
        """
        Biological constraint:
        - oxytocin (idx 2) and cortisol (idx 3) should be anti-correlated
        - dopamine (idx 0) and adrenaline (idx 4) should not both spike together
          (calm reward vs panic are different states)
        """
        oxy   = bio_delta[:, 2]  # oxytocin delta
        cort  = bio_delta[:, 3]  # cortisol delta
        dopa  = bio_delta[:, 0]  # dopamine delta
        adren = bio_delta[:, 4]  # adrenaline delta

        # Penalize same-sign movement (should be opposite)
        oxy_cort_conflict  = torch.relu(oxy * cort).mean()
        dopa_adren_spike   = torch.relu((dopa - 0.2) * (adren - 0.2)).mean()

        return oxy_cort_conflict + dopa_adren_spike

    def forward(self, outputs, targets):
        l_cmd   = self.ce(outputs['command_logits'], targets['command'])
        l_delta = self.mse(outputs['bio_delta'], targets['bio_delta_target'])
        l_homeo = self.homeostasis_penalty(outputs['bio_delta'])
        l_cons  = self.state_consistency_loss(outputs['bio_delta'])

        total = (self.w_cmd   * l_cmd
               + self.w_delta * l_delta
               + self.w_homeo * l_homeo
               + self.w_cons  * l_cons)

        return {'total': total,
                'cmd':   l_cmd.item(),
                'delta': l_delta.item(),
                'homeo': l_homeo.item(),
                'cons':  l_cons.item()}


criterion = AFLossV2()
print('✅ AF Loss v2 ready')
print('   Command CE        (w=1.0)')
print('   Bio Delta MSE     (w=2.0)  ← trains on CHANGE not absolute')
print('   Homeostasis       (w=0.3)  ← prefers small swings')
print('   State Consistency (w=0.5)  ← bio anti-correlation constraints')

## 🚀 Cell 7 — Training Loop v2
> **FIX 5:** Each batch now passes `prev_state` into the model.
> Model is trained to produce correct delta given BOTH text AND current state.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from tqdm.notebook import tqdm
import time

EPOCHS   = 10
LR       = 3e-4
MAX_NORM = 1.0

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = OneCycleLR(
    optimizer, max_lr=LR,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS, pct_start=0.1
)

history = {'train_loss': [], 'val_loss': [], 'cmd_acc': [], 'bio_mae': [], 'cons': []}


def run_epoch(loader, training=True):
    model.train(training)
    total_loss = cmd_correct = bio_mae_sum = n = 0
    cons_sum = 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, leave=False, desc='train' if training else 'val '):
            ids        = batch['input_ids'].to(device)
            mask       = batch['attention_mask'].to(device)
            cmd_t      = batch['command'].to(device)
            bio_t      = batch['bio'].to(device)
            delta_t    = batch['bio_delta_target'].to(device)
            prev_state = batch['prev_state'].to(device)  # ← NEW

            # Forward with state conditioning
            outputs = model(ids, mask, bio_state=prev_state)
            losses  = criterion(outputs, {'command': cmd_t,
                                          'bio_delta_target': delta_t})

            if training:
                optimizer.zero_grad()
                losses['total'].backward()
                nn.utils.clip_grad_norm_(model.parameters(), MAX_NORM)
                optimizer.step()
                scheduler.step()

            # Reconstruct absolute bio from delta + prev state for MAE
            pred_bio = (prev_state + outputs['bio_delta']).clamp(0, 1)

            total_loss  += losses['total'].item() * ids.size(0)
            cmd_correct += (outputs['command_logits'].argmax(1) == cmd_t).sum().item()
            bio_mae_sum += (pred_bio - bio_t).abs().mean().item() * ids.size(0)
            cons_sum    += losses['cons'] * ids.size(0)
            n           += ids.size(0)

    return total_loss/n, cmd_correct/n, bio_mae_sum/n, cons_sum/n


print(f'\n🧠 SEOL AF v2 Training — {EPOCHS} epochs\n')
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc, tr_mae, tr_cons = run_epoch(train_loader, training=True)
    vl_loss, vl_acc, vl_mae, vl_cons = run_epoch(val_loader,   training=False)
    elapsed = time.time() - t0

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['cmd_acc'].append(vl_acc)
    history['bio_mae'].append(vl_mae)
    history['cons'].append(vl_cons)

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'Train: {tr_loss:.4f} | '
          f'Val: {vl_loss:.4f} | '
          f'CmdAcc: {vl_acc*100:.1f}% | '
          f'BioMAE: {vl_mae:.4f} | '
          f'Cons: {vl_cons:.4f} | '
          f'⏱{elapsed:.0f}s')

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        torch.save(model.state_dict(), 'af_v2_best.pt')
        print(f'  💾 Saved!')

print(f'\n✅ Done. Best val loss: {best_val_loss:.4f}')

## 📊 Cell 8 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('SEOL AF v2 — Training Metrics', fontsize=13, fontweight='bold')

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Total Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['cmd_acc'])
axes[1].set_title('Command Accuracy'); axes[1].set_ylim(0,1); axes[1].grid(alpha=0.3)

axes[2].plot(history['bio_mae'])
axes[2].set_title('Bio MAE (absolute)'); axes[2].grid(alpha=0.3)

axes[3].plot(history['cons'])
axes[3].set_title('State Consistency Loss'); axes[3].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('af_v2_curves.png', dpi=150)
plt.show()

## 🧬 Cell 9 — Live Stateful Inference
> **FIX 6 (THE BIG ONE):** AFState now persists across ALL turns.
> Model reads current state, produces delta, state updates, homeostasis runs.
> This is the complete biology → tech copy loop.

In [ ]:
model.load_state_dict(torch.load('af_v2_best.pt', map_location=device))
model.eval()

# ── Persistent AF State (lives across all turns) ───────────────────
af = AFState()


def af_turn(text: str, intensity: float = 1.0) -> dict:
    """
    Complete AF turn:
    1. Read current state
    2. Tokenize input
    3. Model forward with CURRENT STATE injected
    4. Get bio_delta
    5. Apply delta to AFState with momentum
    6. Run homeostasis tick
    7. Determine mode from new state
    """
    # Step 1 — capture state before
    state_before = af.as_vector()

    # Step 2 — tokenize
    enc = tokenizer(
        text, max_length=128,
        padding='max_length', truncation=True, return_tensors='pt'
    )
    ids  = enc['input_ids'].to(device)
    mask = enc['attention_mask'].to(device)
    st   = af.as_tensor().to(device)  # ← inject current state

    # Step 3 — forward
    with torch.no_grad():
        out = model(ids, mask, bio_state=st)

    cmd_id    = out['command_logits'].argmax(1).item()
    bio_delta = out['bio_delta'][0].cpu().tolist()

    # Step 4 — apply delta to live state
    new_bio = [max(0.0, min(1.0, state_before[i] + bio_delta[i])) for i in range(N_BIO)]
    af.apply_delta(new_bio, intensity=intensity)

    # Step 5 — homeostasis
    af.homeostasis_tick()

    return {
        'command':  IDX_TO_CMD[cmd_id],
        'delta':    {BIO_CHANNELS[i]: round(bio_delta[i], 3) for i in range(N_BIO)},
        'mode':     af.current_mode(),
    }


# ── Simulate a multi-turn conversation ────────────────────────────
conversation = [
    "Hey, how are you doing today?",
    "I love you so much, you mean everything to me.",
    "I love you so much, you mean everything to me.",  # repeat → state accumulates
    "Actually I need some space right now.",
    "Stop, you're scaring me, I feel threatened.",
    "It's okay, I was overreacting, I'm fine now.",
    "You did amazing today, I'm so proud of you!",
]

print('=' * 60)
print('  SEOL AF v2 — Multi-Turn Stateful Conversation')
print('  (State persists and accumulates across all turns)')
print('=' * 60)

for i, text in enumerate(conversation):
    result = af_turn(text)
    print(f'\nTurn {i+1}: "{text}"')
    print(f'  Command : {result["command"]}')
    print(f'  Mode    : {result["mode"]}')
    af.display()

## 📦 Cell 9.5 — Install ONNX (required before export)

In [ ]:
!pip install -q onnx onnxscript
import onnx
print(f'✅ onnx version: {onnx.__version__}')

## 💾 Cell 10 — Export ONNX + Rust Blueprint
> Model exported with bio_state as input so Rust can pass AFState on every call.

In [ ]:
import torch.onnx, os

model.eval()
dummy_ids   = torch.zeros(1, 128, dtype=torch.long).to(device)
dummy_mask  = torch.ones(1, 128, dtype=torch.long).to(device)
dummy_state = torch.full((1, N_BIO), 0.5).to(device)  # ← NEW input

torch.onnx.export(
    model,
    (dummy_ids, dummy_mask, dummy_state),
    'seol_af_v2.onnx',
    input_names=['input_ids', 'attention_mask', 'bio_state'],
    output_names=['command_logits', 'bio_delta', 'mode_logits'],
    dynamic_axes={
        'input_ids':      {0: 'batch'},
        'attention_mask': {0: 'batch'},
        'bio_state':      {0: 'batch'},
        'command_logits': {0: 'batch'},
        'bio_delta':      {0: 'batch'},
        'mode_logits':    {0: 'batch'},
    },
    opset_version=14, verbose=False,
)

print(f'✅ ONNX: seol_af_v2.onnx ({os.path.getsize("seol_af_v2.onnx")/1e6:.1f} MB)')

# ── Rust usage blueprint ───────────────────────────────────────────
rust_blueprint = '''
// ─── Rust AF Engine Blueprint ───────────────────────────────────
// In your af_state.rs:

pub struct AFState {
    pub dopamine:   f32,
    pub serotonin:  f32,
    pub oxytocin:   f32,
    pub cortisol:   f32,
    pub adrenaline: f32,
    pub endorphin:  f32,
}

impl AFState {
    const BASELINE: f32 = 0.5;
    const DECAY:    f32 = 0.04;
    const MOMENTUM: f32 = 0.35;

    pub fn as_slice(&self) -> [f32; 6] {
        [self.dopamine, self.serotonin, self.oxytocin,
         self.cortisol, self.adrenaline, self.endorphin]
    }

    pub fn apply_delta(&mut self, delta: &[f32; 6], intensity: f32) {
        let alpha = Self::MOMENTUM * intensity;
        let vals  = self.as_slice();
        // blend with momentum
        self.dopamine   = (vals[0] + delta[0] * alpha).clamp(0.0, 1.0);
        self.serotonin  = (vals[1] + delta[1] * alpha).clamp(0.0, 1.0);
        // ... etc
    }

    pub fn homeostasis_tick(&mut self) {
        // Runs in a Rust thread every N ms
        let decay = Self::DECAY;
        let base  = Self::BASELINE;
        self.dopamine   += decay * (base - self.dopamine);
        self.serotonin  += decay * (base - self.serotonin);
        // ... etc
    }
}

// In inference.rs:
// 1. Load seol_af_v2.onnx with ort crate
// 2. Pass [input_ids, attention_mask, af_state.as_slice()]
// 3. Get bio_delta back
// 4. af_state.apply_delta(&bio_delta, 1.0)
// 5. af_state.homeostasis_tick()  ← runs in separate thread
'''

print(rust_blueprint)

from google.colab import files
files.download('seol_af_v2.onnx')
files.download('af_v2_best.pt')
files.download('af_v2_curves.png')